# Procurement Data Explorer

Explore the procurement dataset, try queries, and discover new eval cases.

In [1]:
from pymongo import MongoClient
import json

client = MongoClient("mongodb://localhost:27017")
db = client["procurement"]
collection = db["purchases"]

print(f"Total documents: {collection.count_documents({})}")

Total documents: 346018


## Schema overview

In [2]:
# Sample one document to see all fields
sample = collection.find_one()
for field, value in sample.items():
    print(f"{field:30s} {type(value).__name__:10s} {str(value)[:60]}")

_id                            ObjectId   697ccb56a9c4584b2f2f0929
Creation Date                  datetime   2013-08-27 00:00:00
Purchase Date                  NoneType   None
Fiscal Year                    str        2013-2014
LPA Number                     str        7-12-70-26
Purchase Order Number          str        REQ0011118
Requisition Number             str        REQ0011118
Acquisition Type               str        IT Goods
Sub-Acquisition Type           NoneType   None
Acquisition Method             str        WSCA/Coop
Sub-Acquisition Method         NoneType   None
Department Name                str        Consumer Affairs, Department of
Supplier Code                  str        1740272.0
Supplier Name                  str        Pitney Bowes
Supplier Qualifications        NoneType   None
Supplier Zip Code              NoneType   None
CalCard                        str        NO
Item Name                      str        USB
Item Description               str        USB
Quan

## Distinct values for key fields

In [3]:
for field in ["Fiscal Year", "Acquisition Type", "CalCard"]:
    vals = collection.distinct(field)
    print(f"{field}: {vals}")

Fiscal Year: ['2012-2013', '2013-2014', '2014-2015']
Acquisition Type: ['IT Goods', 'IT Services', 'IT Telecommunications', 'NON-IT Goods', 'NON-IT Services']
CalCard: ['NO', 'YES']


## Aggregation playground

Try pipelines here. When you find something interesting, copy it into `evaluation.json`.

In [4]:
# Example: top departments by total spend
pipeline = [
    {"$group": {"_id": "$Department Name", "total_spend": {"$sum": "$Total Price"}}},
    {"$sort": {"total_spend": -1}},
    {"$limit": 10},
]
list(collection.aggregate(pipeline))

[{'_id': 'Health Care Services, Department of', 'total_spend': 99759350736.42},
 {'_id': 'Public Health, Department of', 'total_spend': 5621707893.98},
 {'_id': 'Social Services, Department of', 'total_spend': 5565328198.27},
 {'_id': 'Corrections and Rehabilitation, Department of',
  'total_spend': 4711857451.29},
 {'_id': 'State Hospitals, Department of', 'total_spend': 4545650046.42},
 {'_id': 'Transportation, Department of', 'total_spend': 4347882799.66},
 {'_id': 'High Speed Rail Authority, California',
  'total_spend': 3565361682.22},
 {'_id': 'Water Resources, Department of', 'total_spend': 2790266200.9},
 {'_id': 'Correctional Health Care Services', 'total_spend': 2641173667.9},
 {'_id': 'Employment Development Department', 'total_spend': 1724960851.03}]

In [5]:
# Example: orders per fiscal year
pipeline = [
    {"$group": {"_id": "$Fiscal Year", "count": {"$sum": 1}}},
    {"$sort": {"_id": 1}},
]
list(collection.aggregate(pipeline))

[{'_id': '2012-2013', 'count': 108845},
 {'_id': '2013-2014', 'count': 120636},
 {'_id': '2014-2015', 'count': 116537}]

In [6]:
# Example: average order value by acquisition type
pipeline = [
    {"$group": {"_id": "$Acquisition Type", "avg_price": {"$avg": "$Total Price"}}},
    {"$sort": {"avg_price": -1}},
]
list(collection.aggregate(pipeline))

[{'_id': 'NON-IT Services', 'avg_price': 2056842.479325455},
 {'_id': 'IT Services', 'avg_price': 387407.4080236193},
 {'_id': 'IT Telecommunications', 'avg_price': 92560.40775510203},
 {'_id': 'IT Goods', 'avg_price': 30937.21148035363},
 {'_id': 'NON-IT Goods', 'avg_price': 21220.12874347254}]

## Query playground

In [7]:
# Example: find high-value orders
cursor = collection.find(
    {"Total Price": {"$gte": 1_000_000}},
    {"Supplier Name": 1, "Department Name": 1, "Total Price": 1, "_id": 0},
).sort("Total Price", -1).limit(10)

list(cursor)

[{'Department Name': 'Health Care Services, Department of',
  'Supplier Name': 'Delta Dental of California',
  'Total Price': 7337038064.0},
 {'Department Name': 'Health Care Services, Department of',
  'Supplier Name': 'L.A. Care Health Plan',
  'Total Price': 3194190000.0},
 {'Department Name': 'Health Care Services, Department of',
  'Supplier Name': 'County of Los Angeles',
  'Total Price': 3010052803.0},
 {'Department Name': 'Health Care Services, Department of',
  'Supplier Name': 'Health Net Community Solutions, Inc.',
  'Total Price': 2474118000.0},
 {'Department Name': 'Health Care Services, Department of',
  'Supplier Name': 'Health Net Community Solutions, Inc.',
  'Total Price': 2253227000.0},
 {'Department Name': 'Public Health, Department of',
  'Supplier Name': 'Ramsell Public Health Rx',
  'Total Price': 2200000000.0},
 {'Department Name': 'Health Care Services, Department of',
  'Supplier Name': 'Ventura County',
  'Total Price': 1979109000.0},
 {'Department Name': 'He

In [ ]:
collection.aggregate()